In [0]:
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType, DoubleType
)
from pyspark.sql.functions import col, current_date
from delta.tables import DeltaTable

schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("name",        StringType(),  True),
    StructField("email",       StringType(),  True),
    StructField("total_spent", DoubleType(),  True),
    StructField("status",      StringType(),  True),
])

existing_customers = [
    (1, "Alice",  "alice@email.com",  1500.00, "active"),
    (2, "Bob",    "bob@email.com",     320.00, "active"),
    (3, "Carol",  "carol@email.com",  2200.00, "active"),
    (4, "David",  "david@email.com",    89.00, "inactive"),
    (5, "Eve",    "eve@email.com",    4100.00, "active"),
]

df_existing = spark.createDataFrame(existing_customers, schema)

df_existing.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("customers")

print("Target table created — 5 existing customers")
spark.read.table("customers").show()

In [0]:
incoming_data = [
    (1, "Alice",   "alice@email.com",   1750.00, "active"),   # existing — spent more
    (3, "Carol",   "carol_new@email.com", 2500.00, "active"), # existing — email changed
    (4, "David",   "david@email.com",     89.00, "active"),   # existing — now active
    (6, "Frank",   "frank@email.com",    430.00, "active"),   # NEW customer
    (7, "Grace",   "grace@email.com",    899.00, "active"),   # NEW customer
]

df_incoming = spark.createDataFrame(incoming_data, schema)

df_incoming.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("customers_incoming")

print("Incoming batch ready")
df_incoming.show()

In [0]:
%sql
merge into customers as target
using customers_incoming as source
on target.customer_id = source.customer_id

when matched then
  update set
    target.name = source.name,
    target.email = source.email,
    target.total_spent = source.total_spent,
    target.status = source.status

when not matched then
  insert (customer_id, name, email, total_spent, status)
  values (source.customer_id, source.name, source.email, source.total_spent, source.status);

In [0]:
%sql 
select * from customers
order by customer_id

In [0]:
from delta.tables import DeltaTable

target_table = DeltaTable.forName(spark, "customers")

new_batch = [
    (2, "Bob",   "bob_updated@email.com", 650.00, "active"),
    (8, "Henry", "henry@email.com",       275.00, "active"),
]

df_more = spark.createDataFrame(new_batch, schema)

target_table.alias("target").merge(
    df_more.alias("source"),
    "target.customer_id = source.customer_id"
).whenMatchedUpdate(
    condition = "source.total_spent != target.total_spent",
    set = {
        "total_spent" : "source.total_spent",
        "status" : "source.status"
    }
).whenNotMatchedInsertAll().execute()

spark.read.table("customers").filter(col("customer_id").isin(1,5)).show()

In [0]:
more_updates = [
    (1, "Alice", "alice@email.com", 1750.00, "active"),  # nothing changed
    (5, "Eve",   "eve@email.com",   4800.00, "active"),  # total_spent changed
]

df_more = spark.createDataFrame(more_updates, schema)

target_table.alias("target").merge(
    df_more.alias("source"),
    "target.customer_id = source.customer_id"
).whenMatchedUpdate(
    condition= "source.total_spent != target.total_spent",
    set= {
        "total_spent": "source.total_spent",
        "status": "source.status"
    }
).whenNotMatchedInsertAll() \
  .execute()

spark.read.table("customers") \
  .filter(col("customer_id").isin(1,5)) \
  .show()